In [0]:
CREATE SCHEMA IF NOT EXISTS saps_ctes.gold;

/*
Dimensión con la información de los saps

saps_key: La clave subrrogada para los saps
id_saps: El id de los saps
saps: El nombre del saps
barrio: El barrio donde está ubicado el saps
ubicacion: Las calles donde está ubicado el saps
contacto_teléfono: El teléfono de contacto del saps
responsable: La persona responsable a cargo del saps
cargo: El cargo que ostenta la persona responsable
tiv: La fecha inicial desde la cual los datos de un registro son válidos.
tfv: La fecha final en la cual los datos de un registro dejaron de ser válidos.
*/
CREATE TABLE saps_ctes.gold.dim_saps (
    saps_key BIGINT GENERATED ALWAYS AS IDENTITY,
    id_saps INT NOT NULL,
    saps STRING NOT NULL,
    barrio STRING NOT NULL,
    ubicacion STRING NOT NULL,
    contacto_telefono STRING NOT NULL,
    responsable STRING NOT NULL,
    cargo STRING NOT NULL,
    tiv DATE NOT NULL DEFAULT CURRENT_DATE,
    tfv DATE NOT NULL DEFAULT DATE '9999-01-01',
    CONSTRAINT pk_dim_saps PRIMARY KEY (saps_key)
)
TBLPROPERTIES (
  'delta.feature.allowColumnDefaults' = 'supported'
);

/*
Tabla de dimensión con los datos de la patología

patologia_key: La clave subrrogada para la dimensión
patologia_cod: El código de la patología según el cie10
agrupacion_cie10: La descripción de la patología según cie10
patología_desc: Una descripción adicional de la patología usada en los saps
*/
CREATE TABLE saps_ctes.gold.dim_patologia (
    patologia_key BIGINT GENERATED ALWAYS AS IDENTITY,
    patologia_cod STRING NOT NULL,
    agrupacion_cie10 STRING NOT NULL,
    patologia_desc STRING NOT NULL,
    CONSTRAINT pk_dim_patologia PRIMARY KEY (patologia_key)
);

/*
Tabla de dimensiones para los rangos etarios y sexo. Se emplea como una junk dimension

rango_etario_key: La clave subrogada de la dimensión
rango_etario: El rango etario
sexo: El sexo de la persona para el rango etario
*/
CREATE TABLE saps_ctes.gold.dim_rango_etario (
    rango_etario_key BIGINT GENERATED ALWAYS AS IDENTITY,
    id_rango_etario INT NOT NULL,
    rango_etario STRING NOT NULL,
    sexo STRING,
    CONSTRAINT pk_dim_rango_etario PRIMARY KEY (rango_etario_key)
);

/*
La tabla de dimensión con los datos sobre el tipo de vacunas

vacuna_key: La clave subrogada de la tabla
tipo_vacuna: El nombre del tipo de vacuna
*/
CREATE TABLE saps_ctes.gold.dim_vacuna (
    vacuna_key BIGINT GENERATED ALWAYS AS IDENTITY,
    tipo_vacuna STRING NOT NULL,
    CONSTRAINT pk_dim_vacuna PRIMARY KEY (vacuna_key)
);

/*
La tabla con los datos de la dimensión calendario

calendario_key: La clave subrogada de la tabla
fecha: La fecha en cuestión
dia: El día de la fecha en cuestión.
mes: El mes en cuestión.
anio: El anio en cuestión.
nombre_dia: El nombre del día en cuestión
nombre_mes: El nombre del mes
dia_mes: La combinacion del dia y mes para la fecha en cuestión.
mes_anio: La combinación mes y año para la fecha en cuestión.
anio_mes: La combinación del año y mes para le fecha en cuestión.
semana_anio: Indica la semana del año que representa el fecha en cuestión.
trimestre: Indica el trimestre del año para la fecha en cuestión.
es_fin_semana: Indica si es fin de semana.

*/
CREATE TABLE saps_ctes.gold.dim_calendario (
    calendario_key  INT,
    fecha           DATE NOT NULL,

    dia             INT NOT NULL,
    mes             INT NOT NULL,
    anio            INT NOT NULL,

    nombre_dia      STRING,
    nombre_mes      STRING,

    dia_mes         STRING,   -- ej: 15-08
    mes_anio        STRING,   -- ej: 08-2023
    anio_mes        STRING,   -- ej: 2023-08

    semana_anio     INT,
    trimestre       INT,

    es_fin_semana   BOOLEAN DEFAULT FALSE,
    es_fin_mes      BOOLEAN,
    es_fin_anio     BOOLEAN,

    CONSTRAINT pk_dim_calendario PRIMARY KEY (calendario_key)
)
TBLPROPERTIES (
  'delta.feature.allowColumnDefaults' = 'supported'
);

/*
La tabla de hechos para las consultas

consulta_key: La clave subrogada para la tabla
saps_key: La clave subrogada del saps
patologia_key: La clave subrogada de la patología
calendario_key: La clave subrogada del calendario
rango_etario_key: La clave subrogada para el rango etario
consulta_cantidad: la cantidad de consultas realizadas
*/
CREATE TABLE saps_ctes.gold.fact_consulta (
    consulta_key        BIGINT
        GENERATED ALWAYS AS IDENTITY,

    saps_key            INT NOT NULL,
    patologia_key       BIGINT NOT NULL,
    calendario_key      INT NOT NULL,
    rango_etario_key    INT NOT NULL,

    consultas_cantidad  INT NOT NULL,

    CONSTRAINT pk_fact_consulta PRIMARY KEY (consulta_key)
);

/*
La tabla de hechos para las inmunizaciones

inmunizacion_key: La clave subrogada de la tabla de hechos

*/
CREATE TABLE saps_ctes.gold.fact_inmunizacion (
    inmunizacion_key    BIGINT
        GENERATED ALWAYS AS IDENTITY,

    saps_key            INT NOT NULL,
    calendario_key      INT NOT NULL,
    vacuna_key          INT NOT NULL,

    cantidad_vacuna     INT NOT NULL,

    CONSTRAINT pk_fact_inmunizacion PRIMARY KEY (inmunizacion_key)
);

select * from saps_ctes.gold.dim_saps limit 1;